# LaBSE fine-tuning on top-5 and bottom-5 IN22-Gen source languages, with IN22-Conv evaluation

This notebook fine-tunes `sentence-transformers/LaBSE` **separately** on the source-language groups selected from your pre-test ranking.

Your selected groups are:

- **Top 5 source languages:** `asm`, `mai`, `ory`, `npi`, `mal`
- **Bottom 5 source languages:** `snd`, `kas`, `brx`, `mni`, `sat`

Experiment design:

1. **Train** on selected source-language pairs from **IN22-Gen**.
2. **Evaluate** baseline LaBSE and the two fine-tuned models on **IN22-Conv**.
3. Compare baseline vs fine-tuned sensitivity/specificity on unseen conversational-domain sentences.

The notebook trains two independent models:

1. `top5_sources` model
2. `bottom5_sources` model

Each run gets its own output folder, best-validation model, final model, checkpoints, train split, validation split, and evaluation outputs.

**Why IN22-Conv evaluation?** You decided to train on IN22-Gen and evaluate on IN22-Conv, so the final comparison is not on the exact same sentences used for fine-tuning.

## 1. Install dependencies

Run this cell first.  
In Colab, after installing, it is safer to do: **Runtime → Restart runtime**, then continue from the imports cell.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
!pip -q install -U sentence-transformers datasets accelerate transformers huggingface_hub pandas numpy scikit-learn tqdm

## 2. Imports, seed, runtime detection

In [ ]:
import os
import random
import math
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import TranslationEvaluator

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IS_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(get_ipython())
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

print("Device:", DEVICE)
print("Running in Colab:", IS_COLAB)
print("Running in Kaggle:", IS_KAGGLE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU memory GB:", round(total_gb, 2))


## 3. Optional: Mount Google Drive / set output folders

For Kaggle, this will use `/kaggle/working`.  
For Colab, it will try to mount Google Drive. You can also skip Drive and use `/content`.

In [ ]:
USE_GOOGLE_DRIVE = IS_COLAB

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/labse_indic_finetuning")
    except Exception as e:
        print("Drive mount failed. Falling back to /content.")
        print("Error:", e)
        BASE_DIR = Path("/content/labse_indic_finetuning")
elif IS_KAGGLE:
    BASE_DIR = Path("/kaggle/working/labse_indic_finetuning")
else:
    BASE_DIR = Path("./labse_indic_finetuning")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

# Final model used for your benchmark notebook.
MODEL_OUTPUT_DIR = OUTPUT_DIR / "labse_indic_finetuned"

# Best model according to validation evaluator.
BEST_MODEL_DIR = OUTPUT_DIR / "labse_indic_finetuned_best"

# Rolling checkpoints used for crash recovery / resume.
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

LOG_DIR = OUTPUT_DIR / "logs"

for d in [BASE_DIR, DATA_DIR, OUTPUT_DIR, MODEL_OUTPUT_DIR, BEST_MODEL_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Base folder:", BASE_DIR)
print("Data folder:", DATA_DIR)
print("Final model folder:", MODEL_OUTPUT_DIR)
print("Best model folder:", BEST_MODEL_DIR)
print("Checkpoint folder:", CHECKPOINT_DIR)


## 4. Training configuration

This version is configured to run **two separate fine-tuning jobs**:

- one using the top-5 source languages,
- one using the bottom-5 source languages.

For a smoke test, keep `QUICK_TRAIN_N_PER_GROUP = 5000`.
For the full selected IN22-Gen pairs, set `QUICK_TRAIN_N_PER_GROUP = 0`.


In [ ]:
BASE_MODEL_NAME = "sentence-transformers/LaBSE"

MAX_SEQ_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.10

# ------------------------------------------------------------
# Top/bottom source-language groups from your pre-test ranking
# ------------------------------------------------------------
TOP5_SOURCE_LANGS = ["asm", "mai", "ory", "npi", "mal"]
BOTTOM5_SOURCE_LANGS = ["snd", "kas", "brx", "mni", "sat"]

# Train these as two separate models.
RUN_GROUPS = {
    "top5_sources": TOP5_SOURCE_LANGS,
    "bottom5_sources": BOTTOM5_SOURCE_LANGS,
}

# Your ranking used 21 target pairs per source language, i.e. Indic targets excluding the source itself.
# Keep this False to match that setup. Set True only if you also want English as a target.
INCLUDE_ENGLISH_TARGET = False

# Set to 0 for full training for each group.
# For IN22-Gen full group size: 5 source languages × 21 targets × 1024 rows = 107,520 pairs per group.
QUICK_TRAIN_N_PER_GROUP = 5000

# Optional cap per source-target direction before concatenation. 0 means use all rows per direction.
MAX_PAIRS_PER_DIRECTION = 0

# Optional cap for validation retrieval evaluator per group.
VAL_MAX_N = 1000

# If GPU is available, AMP reduces memory usage and can speed up training.
USE_AMP = torch.cuda.is_available()

# -----------------------------
# Checkpointing settings
# -----------------------------
RESUME_FROM_LATEST_CHECKPOINT = True
CHECKPOINTS_PER_EPOCH = 2
MIN_CHECKPOINT_STEPS = 100
CHECKPOINT_SAVE_TOTAL_LIMIT = 20

print("Base model:", BASE_MODEL_NAME)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Top 5 sources:", TOP5_SOURCE_LANGS)
print("Bottom 5 sources:", BOTTOM5_SOURCE_LANGS)
print("Run groups:", list(RUN_GROUPS.keys()))
print("Include English target:", INCLUDE_ENGLISH_TARGET)
print("Quick train N per group:", QUICK_TRAIN_N_PER_GROUP)
print("Max pairs per direction:", MAX_PAIRS_PER_DIRECTION)
print("Use AMP:", USE_AMP)
print("Resume from latest checkpoint:", RESUME_FROM_LATEST_CHECKPOINT)


# -----------------------------
# Evaluation settings
# -----------------------------
# Main protocol for this notebook:
# Train on IN22-Gen, evaluate on IN22-Conv.
EVAL_DATASET_NAME = "ai4bharat/IN22-Conv"
EVAL_DATASET_CONFIG = "all"
EVAL_DATASET_SUITE = "IN22-Conv"

# Evaluate every Indic source language against every other Indic target language.
# This gives the full 22 x 21 IN22-Conv source-target benchmark.
EVAL_SOURCES = None   # None means use all INDIC_LANGS after language definitions are loaded.
EVAL_TARGETS = None   # None means use all INDIC_LANGS after language definitions are loaded.

# Keep False to match your original 22 Indic-language setup.
EVAL_INCLUDE_ENGLISH_TARGET = False

# Set to 0 for full IN22-Conv evaluation. Use e.g. 200 for a quick smoke test.
QUICK_EVAL_N_PER_LANGUAGE = 0
EVAL_BATCH_SIZE = 128

# Evaluate best-validation checkpoints by default.
# Change to "final" if you want final epoch models instead.
EVAL_MODEL_VARIANT = "best"  # "best" or "final"

print("Eval dataset:", EVAL_DATASET_NAME)
print("Eval model variant:", EVAL_MODEL_VARIANT)
print("Quick eval N per language:", QUICK_EVAL_N_PER_LANGUAGE)


## 5. Model/checkpoint helper functions

Each source-language group is trained as a separate model. These helpers load the base LaBSE model or resume from that group's latest checkpoint.


In [ ]:
def find_latest_sentence_transformer_checkpoint(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        return None

    candidates = []
    for p in checkpoint_dir.glob("*"):
        if not p.is_dir():
            continue
        if (p / "modules.json").exists():
            candidates.append(p)

    if not candidates:
        return None

    return max(candidates, key=lambda p: p.stat().st_mtime)


def make_run_dirs(run_name):
    """Create separate output folders for one fine-tuning run."""
    run_base_dir = OUTPUT_DIR / run_name
    run_model_output_dir = run_base_dir / "labse_indic_finetuned_final"
    run_best_model_dir = run_base_dir / "labse_indic_finetuned_best"
    run_checkpoint_dir = run_base_dir / "checkpoints"
    run_log_dir = run_base_dir / "logs"
    run_data_dir = DATA_DIR / run_name

    for d in [run_base_dir, run_model_output_dir, run_best_model_dir, run_checkpoint_dir, run_log_dir, run_data_dir]:
        d.mkdir(parents=True, exist_ok=True)

    return {
        "run_base_dir": run_base_dir,
        "model_output_dir": run_model_output_dir,
        "best_model_dir": run_best_model_dir,
        "checkpoint_dir": run_checkpoint_dir,
        "log_dir": run_log_dir,
        "data_dir": run_data_dir,
    }


def load_model_for_run(run_name, checkpoint_dir):
    latest_checkpoint = find_latest_sentence_transformer_checkpoint(checkpoint_dir)

    if RESUME_FROM_LATEST_CHECKPOINT and latest_checkpoint is not None:
        model_to_load = str(latest_checkpoint)
        print(f"[{run_name}] Resuming from latest checkpoint: {model_to_load}")
    else:
        model_to_load = BASE_MODEL_NAME
        print(f"[{run_name}] Loading base model: {model_to_load}")

    model = SentenceTransformer(model_to_load, device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    return model, model_to_load


## 6. Load IN22-Gen and build top/bottom source-language training pairs

This cell loads `ai4bharat/IN22-Gen` with the `all` configuration and builds parallel pairs for:

- `top5_sources`: `asm`, `mai`, `ory`, `npi`, `mal`
- `bottom5_sources`: `snd`, `kas`, `brx`, `mni`, `sat`

Because `ai4bharat/IN22-Gen` is gated on Hugging Face, you may need to run:

```python
from huggingface_hub import login
login()
```

and accept the dataset terms on Hugging Face before running the loading cell.


In [ ]:
from datasets import load_dataset

# Mapping from your short language IDs to IN22 language-script IDs.
# The benchmark ranking used short IDs; IN22 columns use script-aware IDs.
LANG_TO_IN22 = {
    "asm": "asm_Beng",
    "ben": "ben_Beng",
    "brx": "brx_Deva",
    "doi": "doi_Deva",
    "gom": "gom_Deva",
    "guj": "guj_Gujr",
    "hin": "hin_Deva",
    "kan": "kan_Knda",
    "kas": "kas_Arab",
    "mai": "mai_Deva",
    "mal": "mal_Mlym",
    "mar": "mar_Deva",
    "mni": "mni_Mtei",
    "npi": "npi_Deva",
    "ory": "ory_Orya",
    "pan": "pan_Guru",
    "san": "san_Deva",
    "sat": "sat_Olck",
    "snd": "snd_Arab",
    "tam": "tam_Taml",
    "tel": "tel_Telu",
    "urd": "urd_Arab",
    "eng": "eng_Latn",
}

INDIC_LANGS = [
    "asm", "ben", "brx", "doi", "gom", "guj", "hin", "kan", "kas", "mai", "mal",
    "mar", "mni", "npi", "ory", "pan", "san", "sat", "snd", "tam", "tel", "urd"
]

TARGET_LANGS = INDIC_LANGS.copy()
if INCLUDE_ENGLISH_TARGET:
    TARGET_LANGS = TARGET_LANGS + ["eng"]


def get_split_as_dataframe(dataset_dict):
    """IN22-Gen usually has a 'gen' split. Fall back to the first split if needed."""
    if hasattr(dataset_dict, "keys"):
        split_name = "gen" if "gen" in dataset_dict else list(dataset_dict.keys())[0]
        print("Using split:", split_name)
        return dataset_dict[split_name].to_pandas()
    return dataset_dict.to_pandas()


def sentence_col(short_lang):
    in22_lang = LANG_TO_IN22[short_lang]
    return f"sentence_{in22_lang}"


def build_pairs_for_source_group(in22_df, source_langs, target_langs, run_name):
    records = []

    for src in source_langs:
        src_col = sentence_col(src)
        if src_col not in in22_df.columns:
            raise ValueError(f"Missing source column for {src}: {src_col}")

        for tgt in target_langs:
            if tgt == src:
                continue

            tgt_col = sentence_col(tgt)
            if tgt_col not in in22_df.columns:
                raise ValueError(f"Missing target column for {tgt}: {tgt_col}")

            pair_df = in22_df[["id", src_col, tgt_col]].copy()
            pair_df = pair_df.rename(columns={src_col: "sentence1", tgt_col: "sentence2"})
            pair_df["run_group"] = run_name
            pair_df["source_language"] = src
            pair_df["target_language"] = tgt
            pair_df["direction"] = src + "→" + tgt

            if MAX_PAIRS_PER_DIRECTION and MAX_PAIRS_PER_DIRECTION > 0:
                pair_df = pair_df.sample(
                    n=min(MAX_PAIRS_PER_DIRECTION, len(pair_df)),
                    random_state=SEED,
                )

            records.append(pair_df)

    out = pd.concat(records, ignore_index=True)
    return out


# Optional manual login if needed:
# from huggingface_hub import login
# login()

try:
    raw_dataset = load_dataset("ai4bharat/IN22-Gen", "all")
except Exception as e:
    raise RuntimeError(
        "Could not load ai4bharat/IN22-Gen. This dataset is gated. "
        "Accept the terms on Hugging Face and run huggingface_hub.login() in Colab, then retry."
    ) from e

in22_df = get_split_as_dataframe(raw_dataset)
print("IN22 dataframe shape:", in22_df.shape)
print("Columns:", list(in22_df.columns)[:10], "...")

# Verify columns exist before building pairs.
needed_langs = sorted(set(sum([langs for langs in RUN_GROUPS.values()], []) + TARGET_LANGS))
missing_cols = [sentence_col(lang) for lang in needed_langs if sentence_col(lang) not in in22_df.columns]
if missing_cols:
    raise ValueError(f"Missing expected IN22 sentence columns: {missing_cols}")

train_df_by_group = {}
for run_name, source_langs in RUN_GROUPS.items():
    group_df = build_pairs_for_source_group(
        in22_df=in22_df,
        source_langs=source_langs,
        target_langs=TARGET_LANGS,
        run_name=run_name,
    )
    train_df_by_group[run_name] = group_df
    print(f"{run_name}: {len(group_df):,} raw pairs")
    display(group_df[["source_language", "target_language", "direction", "sentence1", "sentence2"]].head())

# Convenience combined dataframe for inspection only.
train_df = pd.concat(train_df_by_group.values(), ignore_index=True)
print("Combined selected pairs:", len(train_df))


## 7. Dataset sanity check and cleaning

This cleans each run group independently and optionally samples each group for a quick smoke test.


In [ ]:
if "train_df_by_group" not in globals():
    raise ValueError("train_df_by_group is not defined. Run the IN22-Gen loading cell first.")

required_cols = ["sentence1", "sentence2", "source_language", "target_language", "direction", "run_group"]

cleaned_train_df_by_group = {}

for run_name, group_df in train_df_by_group.items():
    missing = [col for col in required_cols if col not in group_df.columns]
    if missing:
        raise ValueError(f"[{run_name}] Missing required columns: {missing}")

    group_df = group_df[required_cols + (["id"] if "id" in group_df.columns else [])].copy()

    group_df["sentence1"] = group_df["sentence1"].astype(str).str.strip()
    group_df["sentence2"] = group_df["sentence2"].astype(str).str.strip()

    group_df = group_df[
        (group_df["sentence1"] != "") &
        (group_df["sentence2"] != "") &
        (group_df["sentence1"].str.lower() != "nan") &
        (group_df["sentence2"].str.lower() != "nan")
    ].copy()

    group_df = group_df.drop_duplicates(
        subset=["source_language", "target_language", "sentence1", "sentence2"]
    ).reset_index(drop=True)

    if QUICK_TRAIN_N_PER_GROUP and QUICK_TRAIN_N_PER_GROUP > 0:
        group_df = group_df.sample(
            n=min(QUICK_TRAIN_N_PER_GROUP, len(group_df)),
            random_state=SEED,
        ).reset_index(drop=True)

    cleaned_train_df_by_group[run_name] = group_df

    print("=" * 80)
    print(run_name)
    print("Pairs after cleaning/sampling:", len(group_df))
    print("Source languages:", sorted(group_df["source_language"].unique()))
    print("Target languages:", len(sorted(group_df["target_language"].unique())))
    print("Directions:", group_df["direction"].nunique())
    display(group_df.groupby(["source_language", "target_language"]).size().reset_index(name="n").head(10))

train_df_by_group = cleaned_train_df_by_group


## 8. Train-validation split

The validation split here is only a training-time sanity check. Your real comparison should still be done by re-running your benchmark after fine-tuning.


In [ ]:
splits_by_group = {}

for run_name, group_df in train_df_by_group.items():
    if len(group_df) < 100:
        raise ValueError(f"[{run_name}] Training data is too small. Add more pairs before fine-tuning.")

    # Stratify by direction when possible so every source-target pair appears in both train and validation.
    stratify_col = group_df["direction"] if group_df["direction"].value_counts().min() >= 2 else None

    train_pairs, val_pairs = train_test_split(
        group_df,
        test_size=0.02,
        random_state=SEED,
        shuffle=True,
        stratify=stratify_col,
    )

    train_pairs = train_pairs.reset_index(drop=True)
    val_pairs = val_pairs.reset_index(drop=True)

    if VAL_MAX_N and len(val_pairs) > VAL_MAX_N:
        val_pairs = val_pairs.sample(n=VAL_MAX_N, random_state=SEED).reset_index(drop=True)

    run_dirs = make_run_dirs(run_name)
    train_pairs_path = run_dirs["data_dir"] / "train_pairs_used.csv"
    val_pairs_path = run_dirs["data_dir"] / "val_pairs_used.csv"

    train_pairs.to_csv(train_pairs_path, index=False)
    val_pairs.to_csv(val_pairs_path, index=False)

    splits_by_group[run_name] = {
        "train_pairs": train_pairs,
        "val_pairs": val_pairs,
        "run_dirs": run_dirs,
        "train_pairs_path": train_pairs_path,
        "val_pairs_path": val_pairs_path,
    }

    print("=" * 80)
    print(run_name)
    print("Train pairs:", len(train_pairs))
    print("Validation pairs:", len(val_pairs))
    print("Saved train split to:", train_pairs_path)
    print("Saved validation split to:", val_pairs_path)


## 9. Fine-tuning setup

`MultipleNegativesRankingLoss` uses in-batch negatives. The actual `InputExample`, dataloader, loss, evaluator, and model are created separately inside each run so the top-5 and bottom-5 models remain independent.


In [ ]:
def make_train_dataloader(train_pairs):
    train_examples = [
        InputExample(texts=[row["sentence1"], row["sentence2"]])
        for _, row in train_pairs.iterrows()
    ]

    train_dataloader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=BATCH_SIZE,
        drop_last=True,
    )

    return train_examples, train_dataloader


def make_validation_evaluator(val_pairs, run_name):
    if len(val_pairs) == 0:
        return None

    return TranslationEvaluator(
        source_sentences=val_pairs["sentence1"].tolist(),
        target_sentences=val_pairs["sentence2"].tolist(),
        name=f"{run_name}_val_translation_retrieval",
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
    )

for run_name, bundle in splits_by_group.items():
    train_examples, train_dataloader = make_train_dataloader(bundle["train_pairs"])
    warmup_steps_preview = math.ceil(len(train_dataloader) * EPOCHS * WARMUP_RATIO)
    print("=" * 80)
    print(run_name)
    print("Train examples:", len(train_examples))
    print("Train batches:", len(train_dataloader))
    print("Warmup steps:", warmup_steps_preview)


## 10. Validation evaluator

The evaluator is created inside each training run. It checks whether each source sentence retrieves its paired target sentence within that run's validation split.


In [ ]:
for run_name, bundle in splits_by_group.items():
    evaluator_preview = make_validation_evaluator(bundle["val_pairs"], run_name)
    print(f"{run_name} evaluator ready:", evaluator_preview is not None)

del evaluator_preview


## 11. Fine-tune LaBSE separately on top-5 and bottom-5 source groups

This cell trains two independent models:

1. `top5_sources`
2. `bottom5_sources`

Each model starts from base LaBSE unless that run has a checkpoint and `RESUME_FROM_LATEST_CHECKPOINT=True`.


In [ ]:
training_summaries = []

for run_name, bundle in splits_by_group.items():
    print("\n" + "#" * 100)
    print(f"STARTING RUN: {run_name}")
    print("#" * 100)

    run_dirs = bundle["run_dirs"]
    train_pairs = bundle["train_pairs"]
    val_pairs = bundle["val_pairs"]

    model, model_to_load = load_model_for_run(run_name, run_dirs["checkpoint_dir"])
    train_examples, train_dataloader = make_train_dataloader(train_pairs)
    train_loss = losses.MultipleNegativesRankingLoss(model)
    evaluator = make_validation_evaluator(val_pairs, run_name)

    steps_per_epoch = len(train_dataloader)
    if steps_per_epoch == 0:
        raise ValueError(f"[{run_name}] No training batches found. Check BATCH_SIZE and training data size.")

    warmup_steps = math.ceil(steps_per_epoch * EPOCHS * WARMUP_RATIO)

    checkpoint_save_steps = max(1, steps_per_epoch // CHECKPOINTS_PER_EPOCH)
    checkpoint_save_steps = max(MIN_CHECKPOINT_STEPS, checkpoint_save_steps)
    checkpoint_save_steps = min(checkpoint_save_steps, steps_per_epoch)

    evaluation_steps = checkpoint_save_steps

    training_config = {
        "run_name": run_name,
        "source_languages": RUN_GROUPS[run_name],
        "target_languages": TARGET_LANGS,
        "base_model_name": BASE_MODEL_NAME,
        "loaded_model": model_to_load,
        "dataset": "ai4bharat/IN22-Gen",
        "dataset_config": "all",
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": warmup_steps,
        "steps_per_epoch": steps_per_epoch,
        "evaluation_steps": evaluation_steps,
        "checkpoint_save_steps": checkpoint_save_steps,
        "checkpoint_save_total_limit": CHECKPOINT_SAVE_TOTAL_LIMIT,
        "quick_train_n_per_group": QUICK_TRAIN_N_PER_GROUP,
        "max_pairs_per_direction": MAX_PAIRS_PER_DIRECTION,
        "train_pairs": len(train_pairs),
        "validation_pairs": len(val_pairs),
        "use_amp": USE_AMP,
        "final_model_dir": str(run_dirs["model_output_dir"]),
        "best_model_dir": str(run_dirs["best_model_dir"]),
        "checkpoint_dir": str(run_dirs["checkpoint_dir"]),
        "train_pairs_path": str(bundle["train_pairs_path"]),
        "val_pairs_path": str(bundle["val_pairs_path"]),
    }

    config_path = run_dirs["run_base_dir"] / "training_config.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(training_config, f, indent=2, ensure_ascii=False)

    print("Run:", run_name)
    print("Sources:", RUN_GROUPS[run_name])
    print("Train pairs:", len(train_pairs))
    print("Validation pairs:", len(val_pairs))
    print("Steps per epoch:", steps_per_epoch)
    print("Evaluation steps:", evaluation_steps)
    print("Checkpoint save steps:", checkpoint_save_steps)
    print("Final model folder:", run_dirs["model_output_dir"])
    print("Best model folder:", run_dirs["best_model_dir"])
    print("Checkpoint folder:", run_dirs["checkpoint_dir"])
    print("Training config saved to:", config_path)

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        evaluator=evaluator,
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        output_path=str(run_dirs["best_model_dir"]),
        optimizer_params={"lr": LEARNING_RATE},
        evaluation_steps=evaluation_steps,
        save_best_model=True,
        show_progress_bar=True,
        use_amp=USE_AMP,
        checkpoint_path=str(run_dirs["checkpoint_dir"]),
        checkpoint_save_steps=checkpoint_save_steps,
        checkpoint_save_total_limit=CHECKPOINT_SAVE_TOTAL_LIMIT,
    )

    model.save(str(run_dirs["model_output_dir"]))
    print(f"[{run_name}] Saved final model to: {run_dirs['model_output_dir']}")
    print(f"[{run_name}] Best validation model at: {run_dirs['best_model_dir']}")

    training_summaries.append(training_config)

    del model, train_loss, train_dataloader, train_examples, evaluator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_path = OUTPUT_DIR / "top_bottom_training_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summaries, f, indent=2, ensure_ascii=False)

print("\nAll requested runs finished.")
print("Summary saved to:", summary_path)


## 12. Output folders

After training, you will have separate folders for the top-5 and bottom-5 source-language models.


In [ ]:
for run_name, bundle in splits_by_group.items():
    run_dirs = bundle["run_dirs"]
    print("=" * 80)
    print(run_name)
    print("Final model:", run_dirs["model_output_dir"])
    print("Best validation model:", run_dirs["best_model_dir"])
    print("Checkpoints:", run_dirs["checkpoint_dir"])
    print("Train split:", bundle["train_pairs_path"])
    print("Validation split:", bundle["val_pairs_path"])


## 13. Quick load test for both fine-tuned models

This tests the final model from each run. Change `model_output_dir` to `best_model_dir` below if you want to test the best-validation checkpoints instead.


In [ ]:
test_sentences = [
    "এইটো এটা পৰীক্ষামূলক বাক্য।",    # Assamese
    "हे एक चाचणी वाक्य आहे.",          # Marathi
    "ᱱᱚᱶᱟ ᱫᱚ ᱢᱤᱫ ᱵᱤᱰᱟᱹᱣ ᱟᱹᱭᱟᱹᱛ ᱠᱟᱱᱟ᱾",  # Santali
    "This is a test sentence.",
]

for run_name, bundle in splits_by_group.items():
    model_dir = bundle["run_dirs"]["model_output_dir"]
    print("=" * 80)
    print("Loading:", run_name)
    print("Model dir:", model_dir)

    finetuned_model = SentenceTransformer(str(model_dir), device=DEVICE)
    finetuned_model.max_seq_length = MAX_SEQ_LENGTH

    emb = finetuned_model.encode(
        test_sentences,
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
    )

    print("Embedding shape:", emb.shape)
    print("Cosine Assamese-Marathi:", float(np.dot(emb[0], emb[1])))
    print("Cosine Assamese-English:", float(np.dot(emb[0], emb[3])))
    print("Cosine Santali-English:", float(np.dot(emb[2], emb[3])))

    del finetuned_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 14. Evaluation pipeline design

For the final comparison, the evaluation pipeline is included **inside this notebook** so you can run everything end-to-end:

1. train `top5_sources` on IN22-Gen,
2. train `bottom5_sources` on IN22-Gen,
3. evaluate base LaBSE, top-5 fine-tuned LaBSE, and bottom-5 fine-tuned LaBSE on IN22-Conv,
4. save pair-level and source-level CSV files.

For a paper/report-quality workflow, it is also reasonable to copy this section into a separate eval-only notebook after the models are trained. The code below is written so it can be reused that way.

## 15. Load IN22-Conv for held-out evaluation

This loads `ai4bharat/IN22-Conv` and prepares the full 22-language evaluation table. Evaluation uses all source languages and all target languages except same-language pairs, unless you change `EVAL_SOURCES` or `EVAL_TARGETS`.

In [ ]:
from datasets import load_dataset


def load_in22_dataset_as_dataframe(dataset_name, dataset_config="all", preferred_splits=("conv", "gen", "test", "validation", "train")):
    """Load an IN22-style dataset and return one pandas DataFrame."""
    try:
        dataset_obj = load_dataset(dataset_name, dataset_config)
    except Exception as e:
        raise RuntimeError(
            f"Could not load {dataset_name} with config={dataset_config!r}. "
            "If the dataset is gated, accept the terms on Hugging Face and run huggingface_hub.login(), then retry."
        ) from e

    if hasattr(dataset_obj, "keys"):
        available_splits = list(dataset_obj.keys())
        chosen_split = None
        for split in preferred_splits:
            if split in available_splits:
                chosen_split = split
                break
        if chosen_split is None:
            chosen_split = available_splits[0]
        print(f"Using split for {dataset_name}: {chosen_split}")
        return dataset_obj[chosen_split].to_pandas()

    return dataset_obj.to_pandas()


def clean_eval_dataframe(eval_df, eval_langs):
    """Keep rows that have non-empty sentences for every evaluation language."""
    eval_df = eval_df.copy()
    needed_cols = [sentence_col(lang) for lang in eval_langs]
    missing_cols = [col for col in needed_cols if col not in eval_df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected sentence columns in eval dataset: {missing_cols}")

    for col in needed_cols:
        eval_df[col] = eval_df[col].astype(str).str.strip()

    mask = np.ones(len(eval_df), dtype=bool)
    for col in needed_cols:
        mask &= eval_df[col].ne("")
        mask &= eval_df[col].str.lower().ne("nan")

    eval_df = eval_df.loc[mask].reset_index(drop=True)

    if QUICK_EVAL_N_PER_LANGUAGE and QUICK_EVAL_N_PER_LANGUAGE > 0:
        eval_df = eval_df.sample(
            n=min(QUICK_EVAL_N_PER_LANGUAGE, len(eval_df)),
            random_state=SEED,
        ).reset_index(drop=True)

    return eval_df


# Resolve evaluation language lists after INDIC_LANGS is defined.
eval_sources = INDIC_LANGS.copy() if EVAL_SOURCES is None else list(EVAL_SOURCES)
eval_targets = INDIC_LANGS.copy() if EVAL_TARGETS is None else list(EVAL_TARGETS)

if EVAL_INCLUDE_ENGLISH_TARGET and "eng" not in eval_targets:
    eval_targets = eval_targets + ["eng"]

eval_langs = sorted(set(eval_sources + eval_targets))

conv_df_raw = load_in22_dataset_as_dataframe(
    dataset_name=EVAL_DATASET_NAME,
    dataset_config=EVAL_DATASET_CONFIG,
    preferred_splits=("conv", "test", "validation", "train"),
)

conv_df = clean_eval_dataframe(conv_df_raw, eval_langs)

print("Raw IN22-Conv shape:", conv_df_raw.shape)
print("Clean IN22-Conv shape:", conv_df.shape)
print("Evaluation source languages:", len(eval_sources), eval_sources)
print("Evaluation target languages:", len(eval_targets), eval_targets)
print("Evaluation rows per language:", len(conv_df))

## 16. Define sensitivity/specificity evaluation functions

For each source-target direction:

- gold cosine = cosine of aligned translations at the same row index,
- random cosine = cosine after cyclically shifting the target-language rows,
- threshold midpoint = average of mean gold cosine and mean random cosine,
- sensitivity = fraction of gold pairs above the threshold,
- specificity = fraction of random pairs below the threshold.

This recreates the kind of table you used for ranking, but now on IN22-Conv.

In [ ]:
import hashlib


def stable_offset(n, seed, src_lang, tgt_lang, model_name):
    """Deterministic non-zero cyclic offset for random negative pairs."""
    if n < 2:
        raise ValueError("Need at least 2 rows to create random negative pairs.")
    key = f"{seed}:{src_lang}:{tgt_lang}:{model_name}".encode("utf-8")
    digest = hashlib.md5(key).hexdigest()
    return (int(digest[:8], 16) % (n - 1)) + 1


def encode_eval_languages(model, eval_df, eval_langs, batch_size=128):
    """Encode each language column once and cache normalized embeddings."""
    embeddings_by_lang = {}

    for lang in eval_langs:
        col = sentence_col(lang)
        sentences = eval_df[col].tolist()
        print(f"Encoding {lang} ({col}) | n={len(sentences):,}")
        embeddings = model.encode(
            sentences,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        )
        embeddings_by_lang[lang] = embeddings.astype(np.float32, copy=False)

    return embeddings_by_lang


def evaluate_embeddings_by_pair(embeddings_by_lang, source_langs, target_langs, model_name, dataset_suite):
    """Compute pair-level cosine and threshold metrics for all source-target directions."""
    records = []

    for src in source_langs:
        src_emb = embeddings_by_lang[src]
        n = src_emb.shape[0]

        for tgt in target_langs:
            if tgt == src:
                continue
            if tgt not in embeddings_by_lang:
                continue

            tgt_emb = embeddings_by_lang[tgt]
            if tgt_emb.shape[0] != n:
                raise ValueError(f"Row count mismatch for {src}->{tgt}: {n} vs {tgt_emb.shape[0]}")

            gold_cos = np.sum(src_emb * tgt_emb, axis=1)

            offset = stable_offset(n, SEED, src, tgt, model_name)
            random_tgt_emb = np.roll(tgt_emb, shift=offset, axis=0)
            random_cos = np.sum(src_emb * random_tgt_emb, axis=1)

            mean_gold = float(np.mean(gold_cos))
            mean_random = float(np.mean(random_cos))
            threshold_midpoint = float((mean_gold + mean_random) / 2.0)
            sensitivity = float(np.mean(gold_cos >= threshold_midpoint))
            specificity = float(np.mean(random_cos < threshold_midpoint))

            records.append({
                "model_name": model_name,
                "dataset_suite": dataset_suite,
                "source_language": src,
                "target_language": tgt,
                "pair": f"{src} → {tgt}",
                "n_eval_pairs": int(n),
                "mean_gold_cosine": mean_gold,
                "mean_random_cosine": mean_random,
                "cosine_gap": mean_gold - mean_random,
                "threshold_midpoint": threshold_midpoint,
                "sensitivity_midpoint": sensitivity,
                "specificity_midpoint": specificity,
                "balanced_accuracy_midpoint": (sensitivity + specificity) / 2.0,
            })

    return pd.DataFrame(records)


def summarize_by_source(eval_results):
    """Average pair-level metrics over targets for each source language."""
    metric_cols = [
        "mean_gold_cosine",
        "mean_random_cosine",
        "cosine_gap",
        "threshold_midpoint",
        "sensitivity_midpoint",
        "specificity_midpoint",
        "balanced_accuracy_midpoint",
    ]

    summary = (
        eval_results
        .groupby(["model_name", "dataset_suite", "source_language"], as_index=False)
        .agg({**{col: "mean" for col in metric_cols}, "target_language": "count"})
        .rename(columns={"target_language": "num_target_pairs"})
    )

    summary["source_group"] = "other"
    summary.loc[summary["source_language"].isin(TOP5_SOURCE_LANGS), "source_group"] = "top5_source"
    summary.loc[summary["source_language"].isin(BOTTOM5_SOURCE_LANGS), "source_group"] = "bottom5_source"

    return summary


def make_delta_table(summary_df, tuned_model_name, baseline_model_name="labse_base"):
    """Compare one fine-tuned model to baseline at source-language level."""
    metric_cols = [
        "mean_gold_cosine",
        "mean_random_cosine",
        "cosine_gap",
        "sensitivity_midpoint",
        "specificity_midpoint",
        "balanced_accuracy_midpoint",
    ]

    base = summary_df[summary_df["model_name"] == baseline_model_name].copy()
    tuned = summary_df[summary_df["model_name"] == tuned_model_name].copy()

    keep_cols = ["source_language", "source_group", "num_target_pairs"] + metric_cols
    merged = tuned[keep_cols].merge(
        base[keep_cols],
        on=["source_language", "source_group", "num_target_pairs"],
        suffixes=("_finetuned", "_baseline"),
        how="inner",
    )

    merged.insert(0, "finetuned_model", tuned_model_name)

    for col in metric_cols:
        merged[f"delta_{col}"] = merged[f"{col}_finetuned"] - merged[f"{col}_baseline"]

    return merged.sort_values("delta_balanced_accuracy_midpoint", ascending=False).reset_index(drop=True)

## 17. Evaluate base LaBSE and the fine-tuned models on IN22-Conv

By default this evaluates:

1. `labse_base`
2. `labse_top5_sources_best`
3. `labse_bottom5_sources_best`

If a fine-tuned model folder does not exist yet, that model is skipped. Run the training cell first, then rerun this section.

In [ ]:
def get_model_dir_for_run(run_name, variant="best"):
    run_dirs = make_run_dirs(run_name)
    if variant == "best":
        return run_dirs["best_model_dir"]
    if variant == "final":
        return run_dirs["model_output_dir"]
    raise ValueError("variant must be 'best' or 'final'")


model_specs = [
    {
        "model_name": "labse_base",
        "model_path": BASE_MODEL_NAME,
        "kind": "baseline",
    },
    {
        "model_name": f"labse_top5_sources_{EVAL_MODEL_VARIANT}",
        "model_path": str(get_model_dir_for_run("top5_sources", EVAL_MODEL_VARIANT)),
        "kind": "finetuned_top5_sources",
    },
    {
        "model_name": f"labse_bottom5_sources_{EVAL_MODEL_VARIANT}",
        "model_path": str(get_model_dir_for_run("bottom5_sources", EVAL_MODEL_VARIANT)),
        "kind": "finetuned_bottom5_sources",
    },
]

all_eval_results = []

for spec in model_specs:
    model_name = spec["model_name"]
    model_path = spec["model_path"]

    if model_path != BASE_MODEL_NAME and not Path(model_path).exists():
        print(f"SKIPPING {model_name}: model folder not found: {model_path}")
        continue

    print("\n" + "#" * 100)
    print("Evaluating:", model_name)
    print("Model path:", model_path)
    print("#" * 100)

    eval_model = SentenceTransformer(model_path, device=DEVICE)
    eval_model.max_seq_length = MAX_SEQ_LENGTH

    embeddings_by_lang = encode_eval_languages(
        model=eval_model,
        eval_df=conv_df,
        eval_langs=eval_langs,
        batch_size=EVAL_BATCH_SIZE,
    )

    result_df = evaluate_embeddings_by_pair(
        embeddings_by_lang=embeddings_by_lang,
        source_langs=eval_sources,
        target_langs=eval_targets,
        model_name=model_name,
        dataset_suite=EVAL_DATASET_SUITE,
    )
    result_df["model_kind"] = spec["kind"]
    all_eval_results.append(result_df)

    print("Pair-level results:", result_df.shape)
    display(result_df.head())

    del eval_model, embeddings_by_lang
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not all_eval_results:
    raise RuntimeError("No models were evaluated. Check model paths and training outputs.")

conv_eval_results = pd.concat(all_eval_results, ignore_index=True)
conv_eval_summary_by_source = summarize_by_source(conv_eval_results)

EVAL_OUTPUT_DIR = OUTPUT_DIR / "conv_eval"
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pair_csv = EVAL_OUTPUT_DIR / "conv_eval_all_models_by_pair.csv"
summary_csv = EVAL_OUTPUT_DIR / "conv_eval_summary_by_source.csv"

conv_eval_results.to_csv(pair_csv, index=False)
conv_eval_summary_by_source.to_csv(summary_csv, index=False)

print("Saved pair-level evaluation to:", pair_csv)
print("Saved source-level summary to:", summary_csv)

display(conv_eval_results.head())
display(conv_eval_summary_by_source.head())

## 18. Compare fine-tuned models against baseline

Positive deltas mean the fine-tuned model improved over base LaBSE on IN22-Conv. The most important columns are:

- `delta_sensitivity_midpoint`
- `delta_specificity_midpoint`
- `delta_balanced_accuracy_midpoint`
- `delta_cosine_gap`

For your pretest, focus especially on whether the **bottom-5 source model** improves `snd`, `kas`, `brx`, `mni`, and `sat`.

In [ ]:
baseline_name = "labse_base"
top5_model_name = f"labse_top5_sources_{EVAL_MODEL_VARIANT}"
bottom5_model_name = f"labse_bottom5_sources_{EVAL_MODEL_VARIANT}"

selected_sources = TOP5_SOURCE_LANGS + BOTTOM5_SOURCE_LANGS

print("Selected source-language summary on IN22-Conv")
selected_summary = conv_eval_summary_by_source[
    conv_eval_summary_by_source["source_language"].isin(selected_sources)
].sort_values(["source_group", "source_language", "model_name"])
display(selected_summary)

if top5_model_name in conv_eval_summary_by_source["model_name"].unique():
    delta_top5 = make_delta_table(
        summary_df=conv_eval_summary_by_source,
        tuned_model_name=top5_model_name,
        baseline_model_name=baseline_name,
    )
    delta_top5_csv = EVAL_OUTPUT_DIR / f"delta_{top5_model_name}_vs_{baseline_name}_by_source.csv"
    delta_top5.to_csv(delta_top5_csv, index=False)
    print("Top-5 fine-tuned model delta saved to:", delta_top5_csv)
    display(delta_top5[delta_top5["source_language"].isin(selected_sources)])
else:
    delta_top5 = None
    print(f"Skipping top-5 delta table because {top5_model_name} was not evaluated.")

if bottom5_model_name in conv_eval_summary_by_source["model_name"].unique():
    delta_bottom5 = make_delta_table(
        summary_df=conv_eval_summary_by_source,
        tuned_model_name=bottom5_model_name,
        baseline_model_name=baseline_name,
    )
    delta_bottom5_csv = EVAL_OUTPUT_DIR / f"delta_{bottom5_model_name}_vs_{baseline_name}_by_source.csv"
    delta_bottom5.to_csv(delta_bottom5_csv, index=False)
    print("Bottom-5 fine-tuned model delta saved to:", delta_bottom5_csv)
    display(delta_bottom5[delta_bottom5["source_language"].isin(selected_sources)])
else:
    delta_bottom5 = None
    print(f"Skipping bottom-5 delta table because {bottom5_model_name} was not evaluated.")

## 19. Optional quick plots of source-level improvement

These plots show improvement in average balanced accuracy across target languages. Positive bars indicate improvement over base LaBSE on IN22-Conv.

In [ ]:
def plot_delta_by_source(delta_df, title, selected_only=True):
    if delta_df is None or len(delta_df) == 0:
        print("No delta dataframe to plot.")
        return

    plot_df = delta_df.copy()
    if selected_only:
        plot_df = plot_df[plot_df["source_language"].isin(selected_sources)].copy()

    plot_df = plot_df.sort_values("delta_balanced_accuracy_midpoint", ascending=True)

    plt.figure(figsize=(10, max(5, 0.45 * len(plot_df))))
    plt.barh(plot_df["source_language"], plot_df["delta_balanced_accuracy_midpoint"])
    plt.axvline(0, linewidth=1)
    plt.xlabel("Δ balanced accuracy on IN22-Conv")
    plt.ylabel("Source language")
    plt.title(title)

    for _, row in plot_df.iterrows():
        plt.text(
            row["delta_balanced_accuracy_midpoint"],
            row["source_language"],
            f" {row['delta_balanced_accuracy_midpoint']:+.4f}",
            va="center",
            fontsize=9,
        )

    plt.tight_layout()
    plt.show()


plot_delta_by_source(delta_top5, f"{top5_model_name} vs base LaBSE", selected_only=True)
plot_delta_by_source(delta_bottom5, f"{bottom5_model_name} vs base LaBSE", selected_only=True)

## 20. What to report

Use the CSV files saved under `outputs/conv_eval/`.

Recommended report tables:

1. **Pair-level table:** `conv_eval_all_models_by_pair.csv`
2. **Source-level summary:** `conv_eval_summary_by_source.csv`
3. **Top-5 model deltas:** `delta_labse_top5_sources_best_vs_labse_base_by_source.csv`
4. **Bottom-5 model deltas:** `delta_labse_bottom5_sources_best_vs_labse_base_by_source.csv`

The main conclusion should be based on IN22-Conv deltas, not the training loss. In particular, check whether fine-tuning the bottom-5 source languages improves `snd`, `kas`, `brx`, `mni`, and `sat` more than it harms other languages.